# DualExpertFusion Research Pipeline — APTOS 2019 Diabetic Retinopathy

This notebook implements a research-grade training and evaluation pipeline for the DualExpertFusion model on the APTOS 2019 dataset, with multiple imbalance-handling strategies.

---

## 1. Import Libraries and Global Configuration

In this section we import core libraries (PyTorch, scikit-learn, imbalanced-learn, pandas, numpy, matplotlib, seaborn, OpenCV) and define global configuration such as paths, image size (224x224), batch size, learning rate, number of epochs, and random seeds.

In [2]:
# 1. Import Libraries and Global Configuration
import os
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

try:
    import cv2
    HAS_CV2 = True
except ImportError:
    HAS_CV2 = False
    print("⚠ OpenCV not installed. CLAHE-based experiments will be skipped.")

try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False
    print("⚠ imbalanced-learn (SMOTE) not installed. SMOTE-based experiments will be skipped.")

# Optional: use existing metrics helper if available
try:
    from metrics import MetricsCalculator
    HAS_METRICS_HELPER = True
except ImportError:
    HAS_METRICS_HELPER = False

# DualExpertFusion model from project
from dual_expert_model import build_dual_expert

# Global configuration
SEED = 42
NUM_CLASSES = 5
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_EPOCHS = 100
BASE_LR = 1e-3
WEIGHT_DECAY = 1e-4

DATA_ROOT = Path("APTOS2019")  # adjust if needed
TRAIN_CSV = DATA_ROOT / "train.csv"  # standard APTOS 2019 CSV
IMAGE_DIR = DATA_ROOT / "train_images"

EXPERIMENTS_ROOT = Path("experiments_dual")
EXPERIMENTS_ROOT.mkdir(parents=True, exist_ok=True)

# Device & reproducibility
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("\n✓ Environment initialized")

Device: cuda

✓ Environment initialized


## 2. Load APTOS 2019 Metadata and Inspect Class Distribution

This section loads the APTOS 2019 training CSV, maps labels to integer classes, and inspects the class distribution.

In [3]:
# 2. Load APTOS 2019 metadata and inspect class distribution
assert TRAIN_CSV.exists(), f"CSV not found: {TRAIN_CSV}"
assert IMAGE_DIR.exists(), f"Image directory not found: {IMAGE_DIR}"

aptos_df = pd.read_csv(TRAIN_CSV)

# Expect columns like 'id_code' and 'diagnosis'
if 'id_code' in aptos_df.columns and 'diagnosis' in aptos_df.columns:
    aptos_df['image_path'] = aptos_df['id_code'].astype(str) + '.png'
    label_col = 'diagnosis'
else:
    # Fallback: assume columns 'image_path' and 'label'
    assert 'image_path' in aptos_df.columns and 'label' in aptos_df.columns, "CSV must contain image and label columns."
    label_col = 'label'

aptos_df['label'] = aptos_df[label_col].astype(int)

# Map to 0..4 (already in APTOS)
num_samples = len(aptos_df)
print(f"Total samples: {num_samples}")

class_counts = aptos_df['label'].value_counts().sort_index()
class_percent = class_counts / num_samples * 100
print("\nClass distribution (full dataset):")
for c in range(NUM_CLASSES):
    count = int(class_counts.get(c, 0))
    pct = class_percent.get(c, 0.0)
    print(f"  Class {c}: {count} ({pct:.1f}%)")

# Bar plot of class distribution
plt.figure(figsize=(6, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette="viridis")
plt.xlabel("DR Grade")
plt.ylabel("Count")
plt.title("APTOS 2019 — Class Distribution (Full Dataset)")
plt.tight_layout()
plt.show()

Total samples: 3662

Class distribution (full dataset):
  Class 0: 1805 (49.3%)
  Class 1: 370 (10.1%)
  Class 2: 999 (27.3%)
  Class 3: 193 (5.3%)
  Class 4: 295 (8.1%)


## 3. Stratified Dataset Split Strategy (80 / 10 / 10)

We compare three options (80/10/10, 80/20 with cross-validation, 70/10/20).
For this notebook we adopt **80 / 10 / 10** stratified splitting:

- 80% for training (used to fit the model).
- 10% for validation (used for early stopping, hyperparameter tuning, and model selection).
- 10% for a dedicated, untouched test set (used only once at the end for final evaluation).

This setup offers a good compromise between training data size and evaluation reliability,
and clearly separates validation (used during training) from the final test set, which helps
avoid subtle forms of data leakage that can occur when cross-validation results are reused
for model selection and reporting.

In [4]:
# 3. Stratified 80/10/10 split
X = aptos_df['image_path'].values
y = aptos_df['label'].values

# First split: 80% train, 20% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Second split: 10% val, 10% test (50/50 of temp)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
)

print(f"Train size: {len(X_train)} ({len(X_train)/len(X):.1%})")
print(f"Val size:   {len(X_val)} ({len(X_val)/len(X):.1%})")
print(f"Test size:  {len(X_test)} ({len(X_test)/len(X):.1%})")

from collections import Counter

def summarize_split(name, labels):
    counts = Counter(labels)
    total = len(labels)
    print(f"\n{name} split class distribution:")
    for c in range(NUM_CLASSES):
        n = counts.get(c, 0)
        print(f"  Class {c}: {n} ({n/total*100:.1f}%)")

summarize_split("Train", y_train)
summarize_split("Val", y_val)
summarize_split("Test", y_test)

Train size: 2929 (80.0%)
Val size:   366 (10.0%)
Test size:  367 (10.0%)

Train split class distribution:
  Class 0: 1444 (49.3%)
  Class 1: 296 (10.1%)
  Class 2: 799 (27.3%)
  Class 3: 154 (5.3%)
  Class 4: 236 (8.1%)

Val split class distribution:
  Class 0: 180 (49.2%)
  Class 1: 37 (10.1%)
  Class 2: 100 (27.3%)
  Class 3: 20 (5.5%)
  Class 4: 29 (7.9%)

Test split class distribution:
  Class 0: 181 (49.3%)
  Class 1: 37 (10.1%)
  Class 2: 100 (27.2%)
  Class 3: 19 (5.2%)
  Class 4: 30 (8.2%)


## 4. Image Preprocessing and Augmentation Pipelines

We keep preprocessing minimal: resize to 224×224, convert to tensor, normalize,
and (optionally) apply random rotation for training and CLAHE for specific experiments.

In [5]:
# 4. Define preprocessing and augmentation transforms
import torchvision.transforms as T
from PIL import Image

# Simple normalization (you may replace with ImageNet stats if desired)
MEAN = [0.5, 0.5, 0.5]
STD = [0.5, 0.5, 0.5]


def get_transforms(apply_rotation: bool = False):
    """Return train and eval transform pipelines."""
    base_transforms = [
        T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD),
    ]

    if apply_rotation:
        train_transforms = [
            T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            T.RandomRotation(degrees=15, fill=(0, 0, 0)),
            T.ToTensor(),
            T.Normalize(mean=MEAN, std=STD),
        ]
    else:
        train_transforms = base_transforms

    train_tf = T.Compose(train_transforms)
    eval_tf = T.Compose(base_transforms)
    return train_tf, eval_tf


# CLAHE is applied in the dataset (raw image) before transforms when enabled.

## 5. Custom Dataset and DataLoader Builders

We define a PyTorch Dataset that loads images from disk, optionally applies CLAHE,
and then applies the chosen transform. Helper functions build DataLoaders for each split.

In [6]:
# 5. Dataset and DataLoader helpers

class APTOSDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None, apply_clahe: bool = False):
        self.image_paths = list(image_paths)
        self.labels = np.array(labels, dtype=np.int64)
        self.transform = transform
        self.apply_clahe = apply_clahe

    def __len__(self):
        return len(self.image_paths)

    def _apply_clahe(self, img_rgb):
        if not HAS_CV2:
            return img_rgb
        lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        cl = clahe.apply(l)
        lab = cv2.merge((cl, a, b))
        return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    def __getitem__(self, idx):
        rel_path = self.image_paths[idx]
        img_path = IMAGE_DIR / rel_path
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            raise FileNotFoundError(f"Could not read image: {img_path}")
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        if self.apply_clahe:
            img_rgb = self._apply_clahe(img_rgb)

        img = Image.fromarray(img_rgb)
        if self.transform is not None:
            img = self.transform(img)

        label = int(self.labels[idx])
        return img, label


def build_dataloaders(apply_rotation: bool, apply_clahe: bool,
                      batch_size: int = BATCH_SIZE,
                      train_paths=None, train_labels=None,
                      val_paths=None, val_labels=None,
                      test_paths=None, test_labels=None,
                      train_sampler=None):
    """Create train/val/test DataLoaders for a given configuration."""
    if train_paths is None:
        train_paths, train_labels = X_train, y_train
        val_paths, val_labels = X_val, y_val
        test_paths, test_labels = X_test, y_test

    train_tf, eval_tf = get_transforms(apply_rotation=apply_rotation)

    train_ds = APTOSDataset(train_paths, train_labels,
                            transform=train_tf, apply_clahe=apply_clahe)
    val_ds = APTOSDataset(val_paths, val_labels,
                          transform=eval_tf, apply_clahe=apply_clahe)
    test_ds = APTOSDataset(test_paths, test_labels,
                           transform=eval_tf, apply_clahe=apply_clahe)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=(train_sampler is None),
        sampler=train_sampler,
        num_workers=0,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

    return train_loader, val_loader, test_loader

## 6. Imbalance Handling Utilities (Class Weights, Weighted Sampler, SMOTE-like Oversampling)

Here we provide utilities for handling class imbalance:
- Per-class weights for loss (for class-weighted cross-entropy).
- A WeightedRandomSampler for oversampling minority classes.
- A practical SMOTE-like option that rebalances the training set via oversampling.

Note: True pixel-level SMOTE for high-resolution images is computationally heavy and can
distort image structure. For simplicity and stability, we approximate SMOTE-based
oversampling using a class-balanced sampler. Feature-space SMOTE (e.g., on deep embeddings)
is recommended for more advanced studies.

In [7]:
# 6. Imbalance handling utilities
from torch.utils.data import WeightedRandomSampler


def compute_class_weights(labels, num_classes: int = NUM_CLASSES):
    labels = np.asarray(labels, dtype=np.int64)
    counts = np.bincount(labels, minlength=num_classes)
    total = counts.sum()
    # Inverse frequency
    weights = total / (num_classes * (counts + 1e-6))
    weights = torch.tensor(weights, dtype=torch.float32, device=device)
    return weights


def create_weighted_sampler(labels, num_classes: int = NUM_CLASSES):
    labels = np.asarray(labels, dtype=np.int64)
    class_weights = compute_class_weights(labels, num_classes=num_classes).cpu().numpy()
    sample_weights = class_weights[labels]
    sample_weights = torch.from_numpy(sample_weights).double()
    sampler = WeightedRandomSampler(weights=sample_weights,
                                    num_samples=len(sample_weights),
                                    replacement=True)
    return sampler


def prepare_smote_like_sampler(labels):
    """Return a sampler that approximates SMOTE-style oversampling via class-balanced sampling."""
    print("Using class-balanced WeightedRandomSampler as a practical SMOTE-like oversampling strategy.")
    return create_weighted_sampler(labels)

## 7. DualExpertFusion Model Integration

We import the DualExpertFusion architecture from the project and define a simple factory
function that returns a fresh instance on the selected device.

In [8]:
# 7. DualExpertFusion model factory

def create_dual_expert(num_classes: int = NUM_CLASSES):
    """Create a fresh DualExpertFusion model instance."""
    model = build_dual_expert(num_classes=num_classes)
    model = model.to(device)
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"DualExpertFusion parameters: {total_params:,}")
    return model

## 8. Loss Functions and Optimizer/Scheduler Factories

We support standard cross-entropy (optionally with class weights) and Focal Loss
for imbalance handling, plus optimizers and learning-rate schedulers.

In [21]:
# 8. Loss functions and optimizer/scheduler factories
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau


class FocalLoss(nn.Module):
    """Multi-class Focal Loss with log_softmax for numerical stability."""
    def __init__(self, gamma: float = 2.0, alpha=None, reduction: str = 'mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha  # optional tensor of shape [C]
        self.reduction = reduction

    def forward(self, logits, targets):
        # logits: (B, C), targets: (B,) integer labels
        log_probs = F.log_softmax(logits, dim=1)
        probs = log_probs.exp()
        targets = targets.long()

        log_p_t = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        p_t = probs.gather(1, targets.unsqueeze(1)).squeeze(1)

        focal_weight = (1.0 - p_t) ** self.gamma
        if self.alpha is not None:
            alpha_t = self.alpha.to(logits.device)[targets]
            focal_weight = focal_weight * alpha_t

        loss = -focal_weight * log_p_t
        if self.reduction == 'mean':
            return loss.mean()
        if self.reduction == 'sum':
            return loss.sum()
        return loss


def build_loss(loss_type: str = 'ce', class_weights=None, gamma: float = 2.0):
    if loss_type == 'focal':
        alpha = class_weights if class_weights is not None else None
        return FocalLoss(gamma=gamma, alpha=alpha)
    else:
        return nn.CrossEntropyLoss(weight=class_weights)


def build_optimizer(model, lr: float = BASE_LR, weight_decay: float = WEIGHT_DECAY):
    return AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)


def build_scheduler(optimizer, mode: str = 'max', factor: float = 0.5, patience: int = 5):
    # Reduce LR when validation metric has stopped improving
    scheduler = ReduceLROnPlateau(optimizer, mode=mode, factor=factor,
                                  patience=patience)
    return scheduler

## 9. Metric Computation Utilities

We compute Accuracy, Precision, Recall, F1-score, Specificity, ROC-AUC, QWK, and Loss
from predicted probabilities and ground-truth labels using scikit-learn.

In [10]:
# 9. Metric computation utilities
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix,
    cohen_kappa_score,
)


def compute_metrics(y_true, y_proba, avg_loss: float, num_classes: int = NUM_CLASSES):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_proba = np.asarray(y_proba, dtype=np.float32)
    y_pred = y_proba.argmax(axis=1)

    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )

    # Specificity (macro over classes)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    tn_fp = cm.sum(axis=0)  # TP+FP per predicted class
    tp_fn = cm.sum(axis=1)  # TP+FN per true class
    total = cm.sum()
    specificities = []
    for i in range(num_classes):
        tp = cm[i, i]
        fp = tn_fp[i] - tp
        fn = tp_fn[i] - tp
        tn = total - tp - fp - fn
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specificities.append(spec)
    specificity = float(np.mean(specificities))

    # ROC-AUC (one-vs-rest)
    try:
        y_true_oh = np.eye(num_classes)[y_true]
        roc_auc = roc_auc_score(y_true_oh, y_proba, multi_class='ovr')
    except ValueError:
        roc_auc = float('nan')

    # Quadratic Weighted Kappa
    try:
        qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    except Exception:
        qwk = float('nan')

    metrics = {
        'loss': float(avg_loss),
        'accuracy': float(acc),
        'precision': float(prec),
        'recall': float(rec),
        'f1_score': float(f1),
        'specificity': float(specificity),
        'roc_auc': float(roc_auc),
        'qwk': float(qwk),
        'confusion_matrix': cm,
    }
    return metrics

## 10. Training Loop with Checkpointing, Early Stopping, Scheduler, and Resume

We implement reusable training/evaluation functions that support:
- Best-model checkpointing (by validation QWK).
- Resume from last checkpoint if available.
- Early stopping on validation QWK.
- Learning-rate scheduling on validation QWK.
- Per-epoch logging of train/validation metrics.

In [11]:
# 10. Training / evaluation utilities
from copy import deepcopy


def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    if is_train:
        model.train()
    else:
        model.eval()

    all_labels = []
    all_probs = []
    running_loss = 0.0
    n_samples = 0

    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        with torch.set_grad_enabled(is_train):
            logits = model(imgs)
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        n_samples += batch_size

        probs = F.softmax(logits.detach(), dim=1)
        all_labels.append(labels.detach().cpu().numpy())
        all_probs.append(probs.cpu().numpy())

    avg_loss = running_loss / max(1, n_samples)
    y_true = np.concatenate(all_labels, axis=0)
    y_proba = np.concatenate(all_probs, axis=0)
    metrics = compute_metrics(y_true, y_proba, avg_loss, num_classes=NUM_CLASSES)
    return metrics, y_true, y_proba


def train_model(experiment_dir, model, train_loader, val_loader,
                criterion, optimizer, scheduler,
                num_epochs: int = NUM_EPOCHS,
                patience: int = 15):
    experiment_dir = Path(experiment_dir)
    experiment_dir.mkdir(parents=True, exist_ok=True)

    best_qwk = -1.0
    best_epoch = 0
    history = {
        'epoch': [],
        'train_loss': [], 'train_accuracy': [], 'train_precision': [], 'train_recall': [],
        'train_f1_score': [], 'train_specificity': [], 'train_roc_auc': [], 'train_qwk': [],
        'val_loss': [], 'val_accuracy': [], 'val_precision': [], 'val_recall': [],
        'val_f1_score': [], 'val_specificity': [], 'val_roc_auc': [], 'val_qwk': [],
    }

    ckpt_path = experiment_dir / 'resume_checkpoint.pth'
    best_path = experiment_dir / 'best_model.pth'

    start_epoch = 0
    epochs_no_improve = 0

    if ckpt_path.exists():
        print("↻ Resuming from checkpoint...")
        ckpt = torch.load(str(ckpt_path), map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        history = ckpt['history']
        start_epoch = ckpt['epoch'] + 1
        best_qwk = ckpt['best_qwk']
        best_epoch = ckpt['best_epoch']
        print(f"Resumed from epoch {start_epoch} (best QWK={best_qwk:.4f})")

    for epoch in range(start_epoch, num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")

        train_metrics, _, _ = run_epoch(model, train_loader, criterion, optimizer)
        val_metrics, _, _ = run_epoch(model, val_loader, criterion, optimizer=None)

        # Scheduler step on validation QWK
        scheduler.step(val_metrics['qwk'])

        # Log history
        history['epoch'].append(epoch + 1)
        for prefix, m in [('train_', train_metrics), ('val_', val_metrics)]:
            for key in ['loss', 'accuracy', 'precision', 'recall', 'f1_score',
                        'specificity', 'roc_auc', 'qwk']:
                history[f'{prefix}{key}'].append(m[key])

        curr_qwk = val_metrics['qwk']
        print(f"  Train: Loss={train_metrics['loss']:.4f}, Acc={train_metrics['accuracy']:.4f}, QWK={train_metrics['qwk']:.4f}")
        print(f"  Val:   Loss={val_metrics['loss']:.4f}, Acc={val_metrics['accuracy']:.4f}, QWK={curr_qwk:.4f}")

        # Checkpoint (resume)
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'history': history,
            'best_qwk': best_qwk,
            'best_epoch': best_epoch,
        }, str(ckpt_path))

        # Best model checkpoint
        if curr_qwk > best_qwk:
            best_qwk = curr_qwk
            best_epoch = epoch + 1
            torch.save({
                'epoch': best_epoch,
                'model_state_dict': deepcopy(model.state_dict()),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_qwk': best_qwk,
            }, str(best_path))
            epochs_no_improve = 0
            print(f"  ✓ New best model (epoch {best_epoch}, QWK={best_qwk:.4f}) saved.")
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {patience} epochs without improvement.")
            break

    # Save history CSV
    import pandas as pd
    hist_df = pd.DataFrame(history)
    hist_df.to_csv(str(experiment_dir / 'training_log.csv'), index=False)

    return history, best_epoch, best_qwk

## 11. Experiment Configurations (Exp2, Exp4, Exp5, Exp7 + Exp7 Focal Grid)

We now define the four base experiment configurations, plus a small Focal Loss grid for Exp7:

- **Exp2**: SMOTE-like oversampling + rotation + DualExpert (cross-entropy).
- **Exp4**: class"+"weight + rotation + DualExpert (class-weighted cross-entropy).
- **Exp5**: rotation + DualExpert + Focal Loss.
- **Exp7**: CLAHE + SMOTE-like oversampling + rotation + DualExpert.
- **Exp7_Focal_g1**: Exp7 + Focal Loss (gamma=1.0, alpha=None).
- **Exp7_Focal_g2**: Exp7 + Focal Loss (gamma=2.0, alpha=None).
- **Exp7_Focal_g2_alpha**: Exp7 + Focal Loss (gamma=2.0, alpha=class-weights).

Each configuration specifies whether to use CLAHE, SMOTE-like oversampling, class weights,
and Focal Loss.

In [ ]:
# 11. Experiment configurations

EXPERIMENT_CONFIGS = {
    'Exp2': {
        'name': 'SMOTE-like + Rotation + DualExpert (CE)',
        'use_smote_like': True,
        'use_class_weights': False,
        'use_clahe': False,
        'loss_type': 'ce',
        'use_rotation': True,
    },
    'Exp4': {
        'name': 'Class-Weight + Rotation + DualExpert (CE)',
        'use_smote_like': False,
        'use_class_weights': True,
        'use_clahe': False,
        'loss_type': 'ce',
        'use_rotation': True,
    },
    'Exp5': {
        'name': 'Rotation + DualExpert + Focal Loss',
        'use_smote_like': False,
        'use_class_weights': False,
        'use_clahe': False,
        'loss_type': 'focal',
        'use_rotation': True,
    },
    'Exp7': {
        'name': 'CLAHE + SMOTE-like + Rotation + DualExpert',
        'use_smote_like': True,
        'use_class_weights': False,
        'use_clahe': True,
        'loss_type': 'ce',
        'use_rotation': True,
    },
    'Exp7_Focal_g1': {
        'name': 'CLAHE + SMOTE-like + Rotation + DualExpert (Focal, gamma=1.0)',
        'use_smote_like': True,
        'use_class_weights': False,
        'use_clahe': True,
        'loss_type': 'focal',
        'focal_gamma': 1.0,
        'use_rotation': True,
    },
    'Exp7_Focal_g2': {
        'name': 'CLAHE + SMOTE-like + Rotation + DualExpert (Focal, gamma=2.0)',
        'use_smote_like': True,
        'use_class_weights': False,
        'use_clahe': True,
        'loss_type': 'focal',
        'focal_gamma': 2.0,
        'use_rotation': True,
    },
    'Exp7_Focal_g2_alpha': {
        'name': 'CLAHE + SMOTE-like + Rotation + DualExpert (Focal, gamma=2.0, alpha=class-weights)',
        'use_smote_like': True,
        'use_class_weights': True,
        'use_clahe': True,
        'loss_type': 'focal',
        'focal_gamma': 2.0,
        'use_rotation': True,
    },
}

print("Defined experiments:")
for key, cfg in EXPERIMENT_CONFIGS.items():
    print(f"  {key}: {cfg['name']}")

Defined experiments:
  Exp2: SMOTE-like + Rotation + DualExpert (CE)
  Exp4: Class-Weight + Rotation + DualExpert (CE)
  Exp5: Rotation + DualExpert + Focal Loss
  Exp7: CLAHE + SMOTE-like + Rotation + DualExpert


## 12. Automatic Experiment Runner

We now implement a function that, given an experiment configuration, builds the
DataLoaders (with rotation/CLAHE/SMOTE-like settings), creates the DualExpertFusion
model and optimization objects, runs training with checkpointing and early stopping,
and logs per-epoch metrics. A higher-level loop runs all four experiments in sequence.

In [ ]:
# 12. Automatic experiment runner
from tqdm.auto import tqdm

RESULTS_CSV = EXPERIMENTS_ROOT / 'results_summary.csv'


def run_single_experiment(exp_key, config):
    print("\n" + "=" * 80)
    print(f"Running {exp_key}: {config['name']}")
    print("=" * 80)

    # Prepare sampler / class weights
    train_sampler = None
    class_weights = None

    if config.get('use_smote_like', False):
        train_sampler = prepare_smote_like_sampler(y_train)

    if config.get('use_class_weights', False):
        class_weights = compute_class_weights(y_train, num_classes=NUM_CLASSES)

    # Build loaders
    train_loader, val_loader, test_loader = build_dataloaders(
        apply_rotation=config.get('use_rotation', False),
        apply_clahe=config.get('use_clahe', False),
        batch_size=BATCH_SIZE,
        train_paths=X_train,
        train_labels=y_train,
        val_paths=X_val,
        val_labels=y_val,
        test_paths=X_test,
        test_labels=y_test,
        train_sampler=train_sampler,
    )

    # Model, loss, optimizer, scheduler
    model = create_dual_expert(num_classes=NUM_CLASSES)
    criterion = build_loss(
        loss_type=config['loss_type'],
        class_weights=class_weights,
        gamma=config.get('focal_gamma', 2.0),
    )
    optimizer = build_optimizer(model, lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    scheduler = build_scheduler(optimizer, mode='max', factor=0.5, patience=5)

    exp_dir = EXPERIMENTS_ROOT / exp_key
    history, best_epoch, best_qwk = train_model(exp_dir, model, train_loader, val_loader,
                                                criterion, optimizer, scheduler,
                                                num_epochs=NUM_EPOCHS, patience=15)

    print(f"\nTraining finished for {exp_key}. Best epoch={best_epoch}, best val QWK={best_qwk:.4f}")
    return history, best_epoch, best_qwk


def run_all_experiments():
    results = []
    for key in EXPERIMENT_CONFIGS.keys():
        history, best_epoch, best_qwk = run_single_experiment(key, EXPERIMENT_CONFIGS[key])
        # Placeholder: test evaluation will be added in the next section
        results.append({
            'experiment': key,
            'name': EXPERIMENT_CONFIGS[key]['name'],
            'best_epoch': best_epoch,
            'best_val_qwk': best_qwk,
        })

    df = pd.DataFrame(results)
    print("\nExperiment summary (validation-level):")
    display(df)
    return df

## 13. Test Set Evaluation per Experiment

We now add a utility to load the best checkpoint for each experiment, evaluate it
on the held-out test set, and compute all requested metrics (Accuracy, Precision,
Recall, F1-score, Specificity, ROC-AUC, QWK, Loss).

In [ ]:
# 13. Test evaluation utilities

def evaluate_on_split(model, loader, criterion):
    model.eval()
    all_labels = []
    all_probs = []
    running_loss = 0.0
    n_samples = 0

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            logits = model(imgs)
            loss = criterion(logits, labels)

            batch_size = labels.size(0)
            running_loss += loss.item() * batch_size
            n_samples += batch_size

            probs = F.softmax(logits, dim=1)
            all_labels.append(labels.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    avg_loss = running_loss / max(1, n_samples)
    y_true = np.concatenate(all_labels, axis=0)
    y_proba = np.concatenate(all_probs, axis=0)
    metrics = compute_metrics(y_true, y_proba, avg_loss, num_classes=NUM_CLASSES)
    return metrics, y_true, y_proba


def evaluate_experiments_on_test():
    all_results = []

    for key, cfg in EXPERIMENT_CONFIGS.items():
        print("\n" + "-" * 80)
        print(f"Evaluating {key} on test set...")
        exp_dir = EXPERIMENTS_ROOT / key
        best_path = exp_dir / 'best_model.pth'
        assert best_path.exists(), f"Best model not found for {key}: {best_path}"

        model = create_dual_expert(num_classes=NUM_CLASSES)
        ckpt = torch.load(str(best_path), map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])

        # Prepare loaders (same settings as during training for this experiment)
        train_sampler = None
        class_weights = None

        if cfg.get('use_smote_like', False):
            train_sampler = prepare_smote_like_sampler(y_train)
        if cfg.get('use_class_weights', False):
            class_weights = compute_class_weights(y_train, num_classes=NUM_CLASSES)

        train_loader, val_loader, test_loader = build_dataloaders(
            apply_rotation=cfg.get('use_rotation', False),
            apply_clahe=cfg.get('use_clahe', False),
            batch_size=BATCH_SIZE,
            train_paths=X_train,
            train_labels=y_train,
            val_paths=X_val,
            val_labels=y_val,
            test_paths=X_test,
            test_labels=y_test,
            train_sampler=train_sampler,
        )

        criterion = build_loss(
            loss_type=cfg['loss_type'],
            class_weights=class_weights,
            gamma=cfg.get('focal_gamma', 2.0),
        )

        # Evaluate on train/val/test to report final metrics
        train_metrics, _, _ = evaluate_on_split(model, train_loader, criterion)
        val_metrics, _, _ = evaluate_on_split(model, val_loader, criterion)
        test_metrics, y_true_test, y_proba_test = evaluate_on_split(model, test_loader, criterion)

        print(f"  Test: Loss={test_metrics['loss']:.4f}, Acc={test_metrics['accuracy']:.4f}, "
              f"F1={test_metrics['f1_score']:.4f}, QWK={test_metrics['qwk']:.4f}")

        row = {
            'model': 'DualExpertFusion',
            'experiment': key,
            'name': cfg['name'],
            'best_epoch': None,  # can be filled from training logs if desired
            'test_loss': test_metrics['loss'],
            'test_accuracy': test_metrics['accuracy'],
            'test_precision': test_metrics['precision'],
            'test_recall': test_metrics['recall'],
            'test_f1': test_metrics['f1_score'],
            'test_specificity': test_metrics['specificity'],
            'test_roc_auc': test_metrics['roc_auc'],
            'test_qwk': test_metrics['qwk'],
        }
        all_results.append(row)

        # Save per-experiment confusion matrix and ROC data for later visualization
        np.save(exp_dir / 'test_confusion_matrix.npy', test_metrics['confusion_matrix'])
        np.save(exp_dir / 'test_y_true.npy', y_true_test)
        np.save(exp_dir / 'test_y_proba.npy', y_proba_test)

    results_df = pd.DataFrame(all_results)
    if RESULTS_CSV.exists():
        base_df = pd.read_csv(str(RESULTS_CSV))
        # Drop any previous DualExpertFusion rows for same experiments
        base_df = base_df[~base_df['experiment'].isin(results_df['experiment'])]
        results_df = pd.concat([base_df, results_df], ignore_index=True)

    results_df.to_csv(str(RESULTS_CSV), index=False)
    print("\n✓ Test results saved to:", RESULTS_CSV)
    display(results_df)
    return results_df

## 14. Visualization Functions (Learning Curves, Confusion Matrix, ROC Curves, Class Distributions, Comparison)

We now define helper functions to visualize per-experiment learning curves, confusion
matrices, ROC curves, class distributions, and a comparison bar chart across experiments.

In [15]:
# 14. Visualization helpers
from sklearn.metrics import roc_curve, auc


def plot_learning_curves(history, title_prefix: str, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    epochs = history['epoch']

    def _plot_metric(metric_name, ylabel, fname, maximize=True):
        train_vals = history[f'train_{metric_name}']
        val_vals = history[f'val_{metric_name}']
        plt.figure(figsize=(8, 5))
        plt.plot(epochs, train_vals, label='Train')
        plt.plot(epochs, val_vals, label='Validation')
        if val_vals:
            if maximize:
                best_idx = int(np.argmax(val_vals))
            else:
                best_idx = int(np.argmin(val_vals))
            plt.axvline(epochs[best_idx], color='g', linestyle='--',
                        label=f'Best (epoch {epochs[best_idx]})')
        plt.xlabel('Epoch')
        plt.ylabel(ylabel)
        plt.title(f'{title_prefix} — {ylabel}')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(out_dir / fname, dpi=300, bbox_inches='tight')
        plt.close()

    _plot_metric('loss', 'Loss', 'loss_curve.png', maximize=False)
    _plot_metric('accuracy', 'Accuracy', 'accuracy_curve.png', maximize=True)
    _plot_metric('f1_score', 'F1-score', 'f1_curve.png', maximize=True)
    _plot_metric('roc_auc', 'ROC-AUC', 'roc_auc_curve.png', maximize=True)
    _plot_metric('qwk', 'QWK', 'qwk_curve.png', maximize=True)


def plot_confusion_matrix(cm, class_names, title: str, out_file: Path):
    plt.figure(figsize=(8, 6))
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=cm, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_file, dpi=300, bbox_inches='tight')
    plt.close()


def plot_roc_curves(y_true, y_proba, class_names, title: str, out_file: Path):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_proba = np.asarray(y_proba, dtype=np.float32)
    num_classes = len(class_names)
    y_true_oh = np.eye(num_classes)[y_true]

    plt.figure(figsize=(8, 6))
    colors = plt.cm.Set1(np.linspace(0, 1, num_classes))
    for i in range(num_classes):
        fpr, tpr, _ = roc_curve(y_true_oh[:, i], y_proba[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, color=colors[i], lw=2,
                 label=f"{class_names[i]} (AUC={roc_auc:.3f})")
    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_file, dpi=300, bbox_inches='tight')
    plt.close()


def plot_class_distribution(labels, title: str, out_file: Path):
    labels = np.asarray(labels, dtype=np.int64)
    counts = np.bincount(labels, minlength=NUM_CLASSES)
    plt.figure(figsize=(6, 4))
    sns.barplot(x=list(range(NUM_CLASSES)), y=counts, palette='viridis')
    plt.xlabel('Class')
    plt.ylabel('Count')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_file, dpi=300, bbox_inches='tight')
    plt.close()


def plot_experiment_comparison(results_df: pd.DataFrame, out_file: Path):
    plt.figure(figsize=(8, 5))
    subset = results_df[['experiment', 'test_accuracy', 'test_f1', 'test_qwk']].copy()
    subset = subset.set_index('experiment')
    subset.plot(kind='bar')
    plt.ylabel('Score')
    plt.title('Experiment Comparison (Accuracy, F1, QWK)')
    plt.xticks(rotation=0)
    plt.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_file, dpi=300, bbox_inches='tight')
    plt.close()

## 15. Aggregate Results and Select Best Experiment

We aggregate the per-experiment test metrics into a comparison table and automatically
identify the best-performing configuration based on QWK (primary) and Accuracy (secondary).

In [16]:
# 15. Aggregate results and select best experiment

def summarize_results():
    assert RESULTS_CSV.exists(), f"Results CSV not found: {RESULTS_CSV}"
    df = pd.read_csv(str(RESULTS_CSV))

    # (Re)compute best_epoch from per-experiment training logs if available
    best_epochs = []
    for _, row in df.iterrows():
        exp_key = row.get('experiment')
        exp_dir = EXPERIMENTS_ROOT / str(exp_key)
        log_path = exp_dir / 'training_log.csv'
        if log_path.exists():
            hist_df = pd.read_csv(str(log_path))
            if 'val_qwk' in hist_df.columns:
                idx = int(hist_df['val_qwk'].idxmax())
                best_ep = int(hist_df.loc[idx, 'epoch'])
            else:
                best_ep = int(hist_df['epoch'].max())
        else:
            best_ep = None
        best_epochs.append(best_ep)

    df['best_epoch'] = best_epochs

    display_cols = [
        'model', 'experiment', 'name', 'best_epoch',
        'test_accuracy', 'test_f1', 'test_qwk', 'test_loss',
    ]
    cols = [c for c in display_cols if c in df.columns]

    print("\nFinal comparison table:")
    display(df[cols])

    # Select best by QWK then Accuracy
    df_sorted = df.sort_values(by=['test_qwk', 'test_accuracy'], ascending=[False, False])
    best = df_sorted.iloc[0]
    print("\nBest experiment (by QWK, then Accuracy):")
    print(f"  Model: {best['model']}")
    print(f"  Experiment: {best['experiment']} — {best['name']}")
    print(f"  Best epoch: {best['best_epoch']}")
    print(f"  Test Accuracy: {best['test_accuracy']:.4f}")
    print(f"  Test F1: {best['test_f1']:.4f}")
    print(f"  Test QWK: {best['test_qwk']:.4f}")
    print(f"  Test Loss: {best['test_loss']:.4f}")

    # Comparison bar chart
    plot_experiment_comparison(df, out_file=EXPERIMENTS_ROOT / 'experiment_comparison.png')
    print("\n✓ Comparison bar chart saved to:", EXPERIMENTS_ROOT / 'experiment_comparison.png')

    return df, best

## 16. Suggestions for Improving Performance on Imbalanced DR Datasets

Below are research-oriented suggestions to further improve performance on imbalanced
diabetic retinopathy datasets beyond the experiments implemented in this notebook.

In [17]:
# 16. Suggestions for further improvements
suggestions = {
    'weighted_sampling': [
        'Use WeightedRandomSampler (already demonstrated) with more aggressive rebalancing',
        'Combine class weights in the loss with sampling for heavily skewed datasets',
    ],
    'test_time_augmentation': [
        'At inference, run multiple augmented views per image (rotations, flips, slight crops)',
        'Average logits/probabilities over views to obtain more robust predictions (TTA)',
    ],
    'label_smoothing': [
        'Replace hard one-hot labels with smoothed distributions (e.g., epsilon=0.1)',
        'This can reduce overconfidence and improve calibration on minority classes',
    ],
    'ensembles': [
        'Train multiple DualExpertFusion models with different seeds or augmentations',
        'Average or weighted-average their logits to improve stability and performance',
    ],
    'feature_space_smote': [
        'Extract deep embeddings (e.g., from a pretrained backbone) and apply SMOTE there',
        'Train a classifier on embeddings or map synthetic embeddings back via generative models',
    ],
}

print("Recommended avenues for further research on imbalanced DR datasets:\n")
for topic, ideas in suggestions.items():
    print(f"- {topic}:")
    for idea in ideas:
        print(f"    • {idea}")
    print()

Recommended avenues for further research on imbalanced DR datasets:

- weighted_sampling:
    • Use WeightedRandomSampler (already demonstrated) with more aggressive rebalancing
    • Combine class weights in the loss with sampling for heavily skewed datasets

- test_time_augmentation:
    • At inference, run multiple augmented views per image (rotations, flips, slight crops)
    • Average logits/probabilities over views to obtain more robust predictions (TTA)

- label_smoothing:
    • Replace hard one-hot labels with smoothed distributions (e.g., epsilon=0.1)
    • This can reduce overconfidence and improve calibration on minority classes

- ensembles:
    • Train multiple DualExpertFusion models with different seeds or augmentations
    • Average or weighted-average their logits to improve stability and performance

- feature_space_smote:
    • Extract deep embeddings (e.g., from a pretrained backbone) and apply SMOTE there
    • Train a classifier on embeddings or map synthetic embe

In [18]:
# Utility: display helper for DataFrames in Jupyter
from IPython.display import display

In [19]:
# Helper: generate all visualizations for each experiment

def generate_visualizations_for_all_experiments():
    class_names = [f"Class {i}" for i in range(NUM_CLASSES)]
    for key, cfg in EXPERIMENT_CONFIGS.items():
        exp_dir = EXPERIMENTS_ROOT / key
        log_path = exp_dir / 'training_log.csv'
        cm_path = exp_dir / 'test_confusion_matrix.npy'
        y_true_path = exp_dir / 'test_y_true.npy'
        y_proba_path = exp_dir / 'test_y_proba.npy'

        if not (log_path.exists() and cm_path.exists() and y_true_path.exists() and y_proba_path.exists()):
            print(f"Skipping {key}: missing logs or test outputs.")
            continue

        hist_df = pd.read_csv(str(log_path))
        history = {col: hist_df[col].tolist() for col in hist_df.columns}
        cm = np.load(cm_path)
        y_true = np.load(y_true_path)
        y_proba = np.load(y_proba_path)

        plots_dir = exp_dir / 'plots'
        title_prefix = f"{key} — {cfg['name']}"

        plot_learning_curves(history, title_prefix, plots_dir)
        plot_confusion_matrix(cm, class_names,
                              title=f"{title_prefix} — Confusion Matrix",
                              out_file=plots_dir / 'confusion_matrix.png')
        plot_roc_curves(y_true, y_proba, class_names,
                        title=f"{title_prefix} — ROC Curves",
                        out_file=plots_dir / 'roc_curves.png')
        plot_class_distribution(y_train,
                                title="Training Class Distribution (Original)",
                                out_file=plots_dir / 'class_distribution.png')

        print(f"✓ Plots saved for {key} in {plots_dir}")

In [22]:
# Orchestration cell (run these steps to execute full pipeline)
if __name__ == '__main__':
    print("Step 1: Running all experiments (training with checkpointing and early stopping)...")
    _ = run_all_experiments()

    print("\nStep 2: Evaluating best checkpoints on the held-out test set...")
    results_df = evaluate_experiments_on_test()

    print("\nStep 3: Generating visualizations (learning curves, confusion matrices, ROC curves, distributions)...")
    generate_visualizations_for_all_experiments()

    print("\nStep 4: Aggregating results and selecting best experiment...")
    _, best_exp = summarize_results()
    print("\nDone.")

Step 1: Running all experiments (training with checkpointing and early stopping)...

Running Exp2: SMOTE-like + Rotation + DualExpert (CE)
Using class-balanced WeightedRandomSampler as a practical SMOTE-like oversampling strategy.
DualExpertFusion parameters: 11,244,713

Epoch 1/100
  Train: Loss=1.4223, Acc=0.3578, QWK=0.2778
  Val:   Loss=1.2119, Acc=0.5219, QWK=0.4718
  ✓ New best model (epoch 1, QWK=0.4718) saved.

Epoch 2/100
  Train: Loss=1.2861, Acc=0.4223, QWK=0.4119
  Val:   Loss=1.1947, Acc=0.5164, QWK=0.4191

Epoch 3/100
  Train: Loss=1.2490, Acc=0.4507, QWK=0.4663
  Val:   Loss=0.9917, Acc=0.5847, QWK=0.6041
  ✓ New best model (epoch 3, QWK=0.6041) saved.

Epoch 4/100
  Train: Loss=1.2595, Acc=0.4510, QWK=0.4685
  Val:   Loss=1.2504, Acc=0.4945, QWK=0.5341

Epoch 5/100
  Train: Loss=1.2142, Acc=0.4585, QWK=0.4788
  Val:   Loss=0.9731, Acc=0.6175, QWK=0.6184
  ✓ New best model (epoch 5, QWK=0.6184) saved.

Epoch 6/100
  Train: Loss=1.1912, Acc=0.4701, QWK=0.4854
  Val:   Los

,experiment,name,best_epoch,best_val_qwk
0,Exp2,SMOTE-like + Rotation + DualExpert (CE),59,0.795067
1,Exp4,Class-Weight + Rotation + DualExpert (CE),51,0.803849
2,Exp5,Rotation + DualExpert + Focal Loss,69,0.773415
3,Exp7,CLAHE + SMOTE-like + Rotation + DualExpert,30,0.822779



Step 2: Evaluating best checkpoints on the held-out test set...

--------------------------------------------------------------------------------
Evaluating Exp2 on test set...
DualExpertFusion parameters: 11,244,713
Using class-balanced WeightedRandomSampler as a practical SMOTE-like oversampling strategy.
  Test: Loss=1.6710, Acc=0.7711, F1=0.5255, QWK=0.8189

--------------------------------------------------------------------------------
Evaluating Exp4 on test set...
DualExpertFusion parameters: 11,244,713
  Test: Loss=2.1241, Acc=0.7875, F1=0.5776, QWK=0.8303

--------------------------------------------------------------------------------
Evaluating Exp5 on test set...
DualExpertFusion parameters: 11,244,713
  Test: Loss=0.6718, Acc=0.7384, F1=0.5142, QWK=0.7939

--------------------------------------------------------------------------------
Evaluating Exp7 on test set...
DualExpertFusion parameters: 11,244,713
Using class-balanced WeightedRandomSampler as a practical SMOTE-li

,model,experiment,name,best_epoch,test_loss,test_accuracy,test_precision,test_recall,test_f1,test_specificity,test_roc_auc,test_qwk
0,DualExpertFusion,Exp2,SMOTE-like + Rotation + DualExpert (CE),None,1.670986,0.771117,0.542152,0.519516,0.525454,0.940634,0.885431,0.818852
1,DualExpertFusion,Exp4,Class-Weight + Rotation + DualExpert (CE),None,2.124145,0.787466,0.593614,0.569367,0.577625,0.945573,0.897598,0.830328
2,DualExpertFusion,Exp5,Rotation + DualExpert + Focal Loss,None,0.671797,0.738420,0.524737,0.509063,0.514184,0.932376,0.878493,0.793870
3,DualExpertFusion,Exp7,CLAHE + SMOTE-like + Rotation + DualExpert,None,1.002819,0.801090,0.621148,0.588625,0.596456,0.947230,0.883016,0.873255



Step 3: Generating visualizations (learning curves, confusion matrices, ROC curves, distributions)...
✓ Plots saved for Exp2 in experiments_dual\Exp2\plots
✓ Plots saved for Exp4 in experiments_dual\Exp4\plots
✓ Plots saved for Exp5 in experiments_dual\Exp5\plots
✓ Plots saved for Exp7 in experiments_dual\Exp7\plots

Step 4: Aggregating results and selecting best experiment...

Final comparison table:


,model,experiment,name,best_epoch,test_accuracy,test_f1,test_qwk,test_loss
0,DualExpertFusion,Exp2,SMOTE-like + Rotation + DualExpert (CE),59,0.771117,0.525454,0.818852,1.670986
1,DualExpertFusion,Exp4,Class-Weight + Rotation + DualExpert (CE),51,0.787466,0.577625,0.830328,2.124145
2,DualExpertFusion,Exp5,Rotation + DualExpert + Focal Loss,69,0.738420,0.514184,0.793870,0.671797
3,DualExpertFusion,Exp7,CLAHE + SMOTE-like + Rotation + DualExpert,30,0.801090,0.596456,0.873255,1.002819



Best experiment (by QWK, then Accuracy):
  Model: DualExpertFusion
  Experiment: Exp7 — CLAHE + SMOTE-like + Rotation + DualExpert
  Best epoch: 30
  Test Accuracy: 0.8011
  Test F1: 0.5965
  Test QWK: 0.8733
  Test Loss: 1.0028

✓ Comparison bar chart saved to: experiments_dual\experiment_comparison.png

Done.
